# Titanic Debug Fix Notebook

This notebook runs the full workflow in one place:
- clean the raw Titanic dataset
- train a corrected Logistic Regression model
- evaluate on the test set
- summarize the bugs fixed from `titanic_debug.py`


## Bugs Fixed

1. `Embarked` remains categorical in the cleaned data, so passing features straight into `LogisticRegression` fails without encoding.
2. The broken script predicts on `X_train` but compares against `y_test`, which makes evaluation invalid and causes a length mismatch.
3. The original split uses `shuffle=False` and ignores the defined random seed, which makes the evaluation less reliable and less reproducible.


In [1]:
from __future__ import annotations

import pandas as pd

RANDOM_STATE = 42
INPUT_PATH = 'titanic.csv'
CLEAN_OUTPUT_PATH = 'titanic_clean.csv'

df = pd.read_csv(INPUT_PATH)

columns_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin']
existing_columns_to_drop = [column for column in columns_to_drop if column in df.columns]
df = df.drop(columns=existing_columns_to_drop)

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode(dropna=True).iloc[0])
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

df.to_csv(CLEAN_OUTPUT_PATH, index=False)

print(f'Cleaned dataset saved to {CLEAN_OUTPUT_PATH}')
df.head()


Cleaned dataset saved to titanic_clean.csv


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,S
1,1,1,1,38.0,1,0,71.2833,C
2,1,3,1,26.0,0,0,7.9250,S
3,1,1,1,35.0,1,0,53.1000,S
4,0,3,0,35.0,0,0,8.0500,S


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET_COLUMN = 'Survived'

df = pd.read_csv(CLEAN_OUTPUT_PATH)
X = df.drop(columns=TARGET_COLUMN)
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

categorical_features = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred),
}

for name, value in metrics.items():
    print(f'{name}: {value:.4f}')


Accuracy: 0.8045
Precision: 0.7931
Recall: 0.6667
F1: 0.7244
